In [82]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import tensorflow.keras.datasets.mnist as mnist

In [83]:
#Working on MNIST dataset
data=mnist.load_data()
(x_train,y_train),(x_test,y_test)=data

In [84]:
x_train.shape, y_train.shape, x_test.shape, y_test.shape

((60000, 28, 28), (60000,), (10000, 28, 28), (10000,))

In [85]:
x_train = x_train.reshape(x_train.shape[0], -1)
x_test = x_test.reshape(x_test.shape[0], -1)
print(x_train.shape, x_test.shape)
x_train = np.array(x_train/255., dtype=np.float32)
x_test = np.array(x_test/255., dtype=np.float32)

(60000, 784) (10000, 784)


In [86]:
x_train

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(60000, 784), dtype=float32)

In [87]:
y_train

array([5, 0, 4, ..., 5, 6, 8], shape=(60000,), dtype=uint8)

In [88]:
#Convert the y column into ont hot encoding 
def one_hot(y):
    num_labels=10
    m=y.shape[0]
    one_hot_Y=np.zeros((m,10))
    one_hot_Y[np.arange(m),y]=1
    return one_hot_Y


In [89]:
y_train=one_hot(y_train)
y_test=one_hot(y_test)
y_train.shape, y_test.shape

((60000, 10), (10000, 10))

In [90]:
#Layer object to handle weights, biases, activation (if any) of a layer
class Layer:
    def __init__(self,hidden_units:int,activation:str=None):
        self.hidden_units=hidden_units
        self.activation=activation
        self.W=None
        self.b=None

    def initialize_params(self,input_neurons,hidden_units):
        self.W=np.random.randn(input_neurons,hidden_units)
        if self.activation=='relu':
            #He Initialization
            self.W=self.W*np.sqrt(2/input_neurons)
        elif self.activation=='sigmoid':
            #Xavier Initialization
            self.W=self.W*np.sqrt(1/input_neurons)
        self.b=np.zeros((1,hidden_units))

    def forward_prop(self,x):
        self.input=np.array(x,copy=True)
        if self.W is None:
            self.initialize_params(self.input.shape[-1],self.hidden_units)
        self.Z=np.dot(x,self.W)+self.b
        if self.activation is not None:
            self.A=self.activation_fn(self.Z)
            return self.A
        return self.Z
    
    def activation_fn(self,z,derivative=False):
        if self.activation=='relu':
            return self.deri_relu(z) if derivative else self.relu(z)
        if self.activation=='sigmoid':
            return self.deri_sigmoid(z) if derivative else self.sigmoid(z)
        if self.activation=='softmax':
            return self.softmax(z)

    def relu(self,z):
        return np.maximum(0,z)
    
    def deri_relu(self,z):
        return (z>0).astype(float)
    
    def sigmoid(self,z):
        return 1/(1+np.exp(-z))
    
    def deri_sigmoid(self,z):
        sig=self.sigmoid(z)
        return sig*(1-sig)
    
    def softmax(self,z):
        exp=np.exp(z-np.max(z,axis=1,keepdims=True))
        return exp/np.sum(exp,axis=1,keepdims=True)

    #In case of DL on applying cross entropy and softmax the relationship simplifies to dz=ypred-y,So that's why we don't have to compute the derivative of softmax.

    def __repr__(self):
        return str(f'''Hidden Units={self.hidden_units}; Activation={self.activation}''')

In [91]:
import numpy as np

class Model:
    def __init__(self):
        self.layers = dict()
        self.cache = dict()
        self.grads = dict()
        self.global_step = 0 

    def add(self, layer):
        self.layers[len(self.layers) + 1] = layer

    def set_config(self, learning_rate, epochs, optimizer=None):
        self.epochs = epochs
        self.learning_rate = learning_rate
        self.optimizer = optimizer
        
        if self.optimizer is not None:
            # Initialize optimizer with layers to set up m and v tensors
            self.optimizer.config(self.layers)
            self.optimizer.epochs = self.epochs
            self.optimizer.learning_rate = self.learning_rate

    def forward(self, x):
        current_a = x
        for idx, layer in self.layers.items():
            current_a = layer.forward_prop(current_a)
            self.cache[f'Z{idx}'] = layer.Z
            self.cache[f'A{idx}'] = current_a
        return current_a
        
    def backward_prop(self, y):
        self.grads = {}
        last_layer_idx = max(self.layers.keys())
        m = y.shape[0]

        for idx in reversed(range(1, last_layer_idx + 1)):
            if idx == last_layer_idx:
                # Assuming Cross-Entropy + Softmax derivative
                self.grads[f'dZ{idx}'] = self.cache[f'A{idx}'] - y
            else:
                # Backprop through layers
                self.grads[f'dZ{idx}'] = np.dot(self.grads[f'dZ{idx+1}'], self.layers[idx+1].W.T) * \
                                         self.layers[idx].activation_fn(self.cache[f'Z{idx}'], derivative=True)

            # dW = (1/m) * prev_A.T @ dZ
            # If idx=1, prev_A is the input data
            prev_a = self.layers[idx].input 
            self.grads[f'dW{idx}'] = (1/m) * np.dot(prev_a.T, self.grads[f'dZ{idx}'])
            self.grads[f'db{idx}'] = (1/m) * np.sum(self.grads[f'dZ{idx}'], axis=0, keepdims=True)

    def update_params(self, epoch_num):
        # Increment global step for adaptive optimizers
        self.global_step += 1
        
        for idx in self.layers.keys():
            if self.optimizer is None:
                # Default Vanilla SGD
                self.layers[idx].W -= self.learning_rate * self.grads[f'dW{idx}']
                self.layers[idx].b -= self.learning_rate * self.grads[f'db{idx}']
            else:
                # Pass global_step instead of batch-specific step
                self.optimizer.optimize(idx, self.layers, self.grads, epoch_num, self.global_step)

    def fit(self, x_train, y_train, x_test=None, y_test=None, batch_size=32):
        self.history = {'train_loss': [], 'train_acc': [], 'val_acc': []}
        self.global_step = 0 # Reset step count at start of training

        for i in range(1, self.epochs + 1):
            m = x_train.shape[0]
            perm = np.random.permutation(m)
            x_shuffled = x_train[perm]
            y_shuffled = y_train[perm]

            batches = self.create_batches(x_shuffled, y_shuffled, batch_size)
            epoch_loss = []

            for x, y in batches:
                preds = self.forward(x)
                loss = self.compute_loss(y, preds)
                epoch_loss.append(loss)

                self.backward_prop(y)
                self.update_params(i) # Passes current epoch for Demon/Warmup

            # Metrics and Logging
            avg_loss = np.mean(epoch_loss)
            self.history['train_loss'].append(avg_loss)

            train_pred = self.forward(x_train)
            train_acc = np.mean(np.argmax(train_pred, axis=1) == np.argmax(y_train, axis=1)) * 100
            self.history['train_acc'].append(train_acc)

            log_msg = f'Epoch {i}/{self.epochs} - Loss: {avg_loss:.4f} - Train Acc: {train_acc:.2f}%'
            
            if x_test is not None:
                test_preds = self.forward(x_test)
                val_acc = np.mean(np.argmax(test_preds, axis=1) == np.argmax(y_test, axis=1)) * 100
                self.history['val_acc'].append(val_acc)
                log_msg += f' - Val Acc: {val_acc:.2f}%'
            
            print(log_msg)

    def compute_loss(self, y, y_hat):
        m = y.shape[0]
        y_hat = np.clip(y_hat, 1e-9, 1 - 1e-9)
        return -1/m * np.sum(y * np.log(y_hat))

    def create_batches(self, x, y, batch_size):
        return [(x[i:i+batch_size], y[i:i+batch_size]) for i in range(0, x.shape[0], batch_size)]

    def __repr__(self):
        return f"Model with {len(self.layers)} layers: {list(self.layers.values())}"

In [92]:
class Optimizer:
    def __init__(self, learning_rate=None, name=None, epochs=100):
        self.learning_rate = learning_rate
        self.name = name
        self.epochs = epochs  

    def config(self, layers):
        pass

    def optimize(self, idx, layers: list, grads: dict, *args):
        pass 


class SGDM(Optimizer):
    def __init__(self,learning_rate=0.01,mu_init=0.5,max_mu=0.99,demon=False,beta_init=0.9,warmup=True,**kwargs):
        super().__init__(learning_rate=learning_rate, **kwargs)
        self.mu_init=mu_init
        self.max_mu=max_mu
        self.demon=demon
        self.warmup=warmup
        self.beta_init=beta_init   
        self.m=dict()

    def config(self,layers):
        for i in layers.keys():
            self.m[f'W{i}']=np.zeros_like(layers[i].W)
            self.m[f'b{i}']=np.zeros_like(layers[i].b)
    
    def optimize(self,idx,layers,grads,epoch_num,steps):
        if self.demon:
            p_t=max(0,1-epoch_num/self.epochs)  
            mu=self.beta_init*p_t/((1-self.beta_init)+self.beta_init*p_t)
        elif self.warmup:
            mu=self.max_mu*(1-np.exp(-0.1*epoch_num))
        else:
            mu=self.mu_init

        self.m[f'W{idx}']=mu*self.m[f'W{idx}']-self.learning_rate*grads[f'dW{idx}']
        self.m[f'b{idx}']=mu*self.m[f'b{idx}']-self.learning_rate*grads[f'db{idx}']

        layers[idx].W+=self.m[f'W{idx}']
        layers[idx].b+=self.m[f'b{idx}']


class Nesterov(SGDM):
    def __init__(self,learning_rate,**kwargs):
        super().__init__(learning_rate=learning_rate, **kwargs)

    def optimize(self,idx,layers,grads,epoch_num,steps):
        if self.demon:
            p_t=max(0,1-epoch_num/self.epochs)
            mu=self.beta_init*p_t/((1-self.beta_init)+self.beta_init*p_t)
        elif self.warmup:
            mu=self.max_mu*(1-np.exp(-0.1*epoch_num))
        else:
            mu=self.mu_init
        
        mW_prev=self.m[f'W{idx}'].copy()
        mb_prev=self.m[f'b{idx}'].copy()

        self.m[f'W{idx}']=mu*self.m[f'W{idx}']-self.learning_rate*grads[f'dW{idx}']
        self.m[f'b{idx}']=mu*self.m[f'b{idx}']-self.learning_rate*grads[f'db{idx}']

        w_update=-mu*mW_prev+(1+mu)*self.m[f'W{idx}']
        b_update=-mu*mb_prev+(1+mu)*self.m[f'b{idx}']

        layers[idx].W+=w_update
        layers[idx].b+=b_update


class Adagrad(Optimizer):
    def __init__(self, epsilon=1e-8, **kwargs):
        super().__init__(**kwargs)
        self.epsilon = epsilon
        self.v = dict()

    def optimize(self, idx, layers, grads, epoch_num, steps):
        if f'W{idx}' not in self.v:
            self.v[f'W{idx}'] = np.zeros_like(layers[idx].W)
            self.v[f'b{idx}'] = np.zeros_like(layers[idx].b)

        self.v[f'W{idx}'] += grads[f'dW{idx}']**2
        self.v[f'b{idx}'] += grads[f'db{idx}']**2

        layers[idx].W -= self.learning_rate * grads[f'dW{idx}'] / (np.sqrt(self.v[f'W{idx}']) + self.epsilon)
        layers[idx].b -= self.learning_rate * grads[f'db{idx}'] / (np.sqrt(self.v[f'b{idx}']) + self.epsilon)


class RMSprop(Optimizer):
    def __init__(self, decay_rate=0.9, epsilon=1e-8, **kwargs):
        super().__init__(**kwargs)
        self.decay_rate = decay_rate
        self.epsilon = epsilon
        self.v = dict() 

    def config(self, layers):
        for i in layers.keys():
            self.v[f'W{i}']=np.zeros_like(layers[i].W)
            self.v[f'b{i}']=np.zeros_like(layers[i].b)

    def optimize(self, idx, layers, grads, epoch_num, steps):
        self.v[f'W{idx}']=self.decay_rate*self.v[f'W{idx}']+(1-self.decay_rate)*grads[f'dW{idx}']**2
        self.v[f'b{idx}']=self.decay_rate*self.v[f'b{idx}']+(1-self.decay_rate)*grads[f'db{idx}']**2

        layers[idx].W-=self.learning_rate*grads[f'dW{idx}']/(np.sqrt(self.v[f'W{idx}']+self.epsilon)) 
        layers[idx].b-=self.learning_rate*grads[f'db{idx}']/(np.sqrt(self.v[f'b{idx}']+self.epsilon))


class Adam(Optimizer):
    def __init__(self,learning_rate=0.001,beta1=0.9,beta2=0.999,epsilon=1e-8,weight_decay=False,gamma_init=0.00001,decay_rate=0.8,demon=False,**kwargs):
        super().__init__(learning_rate=learning_rate, **kwargs)
        self.beta1=beta1
        self.beta2=beta2
        self.epsilon=epsilon
        self.weight_decay=weight_decay
        self.gamma_init=gamma_init
        self.decay_rate=decay_rate
        self.demon=demon
        self.m=dict()
        self.v=dict()

    def config(self,layers):
        for i in layers.keys():
            self.m[f'W{i}']=np.zeros_like(layers[i].W)
            self.m[f'b{i}']=np.zeros_like(layers[i].b)
            self.v[f'W{i}']=np.zeros_like(layers[i].W)
            self.v[f'b{i}']=np.zeros_like(layers[i].b)

    def optimize(self, idx, layers, grads, epoch_num, steps):
        dW,db=grads[f'dW{idx}'], grads[f'db{idx}']

        b1=self.beta1
        if self.demon:
            p_t=max(0,1-epoch_num/self.epochs)
            b1=self.beta1*(p_t/(1-self.beta1+self.beta1*p_t))

        t=steps+1   

        self.m[f'W{idx}']=b1*self.m[f'W{idx}']+(1-b1)*dW
        self.m[f'b{idx}']=b1*self.m[f'b{idx}']+(1-b1)*db

        self.v[f'W{idx}']=self.beta2*self.v[f'W{idx}']+(1-self.beta2)*(dW**2)
        self.v[f'b{idx}']=self.beta2*self.v[f'b{idx}']+(1-self.beta2)*(db**2)

        mt_w=self.m[f'W{idx}']/(1-b1**t)
        vt_w=self.v[f'W{idx}']/(1-self.beta2**t)
        mt_b=self.m[f'b{idx}']/(1-b1**t)
        vt_b=self.v[f'b{idx}']/(1-self.beta2**t)

        w_update=-self.learning_rate*mt_w/(np.sqrt(vt_w)+self.epsilon)
        b_update=-self.learning_rate*mt_b/(np.sqrt(vt_b)+self.epsilon)

        if self.weight_decay:
            gamma=self.gamma_init*(self.decay_rate**(epoch_num//5))
            w_update-=gamma*layers[idx].W

        layers[idx].W+=w_update
        layers[idx].b+=b_update


class Nadam(Adam):
    def __init__(self, learning_rate,**kwargs):
        super().__init__(learning_rate=learning_rate, **kwargs)

    def optimize(self, idx, layers, grads, epoch_num, steps):
        dW,db=grads[f'dW{idx}'], grads[f'db{idx}']

        b1=self.beta1
        if self.demon:
            p_t=max(0,1-epoch_num/self.epochs)
            b1=self.beta1*(p_t/(1-self.beta1+self.beta1*p_t))

        t=steps+1   

        self.m[f'W{idx}']=b1*self.m[f'W{idx}']+(1-b1)*dW
        self.m[f'b{idx}']=b1*self.m[f'b{idx}']+(1-b1)*db
        self.v[f'W{idx}']=self.beta2*self.v[f'W{idx}']+(1-self.beta2)*(dW**2)
        self.v[f'b{idx}']=self.beta2*self.v[f'b{idx}']+(1-self.beta2)*(db**2)

        mt_w=self.m[f'W{idx}']/(1-b1**t)
        vt_w=self.v[f'W{idx}']/(1-self.beta2**t)
        mt_b=self.m[f'b{idx}']/(1-b1**t)
        vt_b=self.v[f'b{idx}']/(1-self.beta2**t)

        m_nesterov_w=(b1*mt_w)+((1-b1)*dW/(1-b1**t))
        m_nesterov_b=(b1*mt_b)+((1-b1)*db/(1-b1**t))

        w_update=-self.learning_rate*m_nesterov_w/(np.sqrt(vt_w)+self.epsilon)
        b_update=-self.learning_rate*m_nesterov_b/(np.sqrt(vt_b)+self.epsilon)

        if self.weight_decay:
            gamma=self.gamma_init*(self.decay_rate**(epoch_num//5))
            w_update-=gamma*layers[idx].W

        layers[idx].W+=w_update
        layers[idx].b+=b_update


class Adamax(Adam):
    def __init__(self, learning_rate,**kwargs):
        super().__init__(learning_rate=learning_rate, **kwargs)

    def optimize(self, idx, layers, grads, epoch_num, steps):
        dW,db=grads[f'dW{idx}'], grads[f'db{idx}']

        b1=self.beta1
        if self.demon:
            p_t=max(0,1-epoch_num/self.epochs)
            b1=self.beta1*(p_t/(1-self.beta1+self.beta1*p_t))

        t=steps+1 

        self.m[f'W{idx}']=b1*self.m[f'W{idx}']+(1-b1)*dW
        self.m[f'b{idx}']=b1*self.m[f'b{idx}']+(1-b1)*db

        self.v[f'W{idx}']=np.maximum(self.beta2*self.v[f'W{idx}'],np.abs(dW))
        self.v[f'b{idx}']=np.maximum(self.beta2*self.v[f'b{idx}'],np.abs(db))

        mt_w=self.m[f'W{idx}']/(1-b1**t)
        mt_b=self.m[f'b{idx}']/(1-b1**t)

        w_update=-self.learning_rate*mt_w/(self.v[f'W{idx}']+self.epsilon)
        b_update=-self.learning_rate*mt_b/(self.v[f'b{idx}']+self.epsilon)

        if self.weight_decay:
            gamma=self.gamma_init*(self.decay_rate**(epoch_num//5))
            w_update-=gamma*layers[idx].W

        layers[idx].W+=w_update
        layers[idx].b+=b_update

In [93]:
# Parameters (Ensure these are defined)
lr = 0.001
epochs = 25
batch_size = 32
def run_model(optimizer=None):
    model = Model()
    model.add(Layer(128, activation='relu'))
    model.add(Layer(64, activation='relu'))
    model.add(Layer(10, activation='softmax'))
    model.set_config(epochs=epochs, learning_rate=lr, optimizer=optimizer)

    model.fit(x_train, y_train, x_test, y_test, batch_size=batch_size)
    return model.history                                                                                                                                   

# Initializing Instances (Using local classes, no 'opt.' prefix)
sgdm = SGDM(lr, name='SGDM')
demonSGDM = SGDM(lr, demon=True, name='DemonSGDM')
nesterov = Nesterov(lr, name='Nesterov')
demonNesterov = Nesterov(lr, demon=True, name='DemonNesterov')

adagrad = Adagrad(lr, name='Adagrad')
rmsprop = RMSprop(lr, name='RMSprop')

adam = Adam(lr, name='Adam')
adamW = Adam(lr, weight_decay=True, name='AdamW')
demonAdam = Adam(lr, demon=True, name='DemonAdam')
demonAdamW = Adam(lr, weight_decay=True, demon=True, name='DemonAdamW')

nadam = Nadam(lr, name='NAdam')
nadamW = Nadam(lr, weight_decay=True, name='NAdamW')
demonNAdam = Nadam(lr, demon=True, name='DemonNAdam')
demonNAdamW = Nadam(lr, weight_decay=True, demon=True, name='DemonNAdamW')

adamax = Adamax(lr, name='Adamax')
adamaxW = Adamax(lr, weight_decay=True, name='AdamaxW')
demonAdamax = Adamax(lr, demon=True, name='DemonAdamax')
demonAdamaxW = Adamax(lr, weight_decay=True, demon=True, name='DemonAdamaxW')

In [94]:
optimizers_to_run = [
    sgdm, demonSGDM, nesterov, demonNesterov,
    adagrad, rmsprop,
    adam, adamW, demonAdam, demonAdamW,
    nadam, nadamW, demonNAdam, demonNAdamW,
    adamax, adamaxW, demonAdamax, demonAdamaxW
]

# Dictionary to store the history of each run
comparison_results = {}

for opt in optimizers_to_run:
    print(f"\n" + "="*40)
    print(f"🚀 Training with Optimizer: {opt.name}")
    print("="*40)
    
    # run_model returns model.history
    history = run_model(optimizer=opt)
    comparison_results[opt.name] = history


🚀 Training with Optimizer: SGDM
Epoch 1/25 - Loss: 0.7214 - Train Acc: 87.93% - Val Acc: 88.49%
Epoch 2/25 - Loss: 0.3506 - Train Acc: 91.22% - Val Acc: 91.62%
Epoch 3/25 - Loss: 0.2815 - Train Acc: 92.68% - Val Acc: 92.68%
Epoch 4/25 - Loss: 0.2425 - Train Acc: 93.63% - Val Acc: 93.36%
Epoch 5/25 - Loss: 0.2145 - Train Acc: 94.32% - Val Acc: 93.98%
Epoch 6/25 - Loss: 0.1927 - Train Acc: 94.95% - Val Acc: 94.33%
Epoch 7/25 - Loss: 0.1750 - Train Acc: 95.33% - Val Acc: 94.85%
Epoch 8/25 - Loss: 0.1597 - Train Acc: 95.76% - Val Acc: 95.03%
Epoch 9/25 - Loss: 0.1465 - Train Acc: 96.16% - Val Acc: 95.38%
Epoch 10/25 - Loss: 0.1353 - Train Acc: 96.45% - Val Acc: 95.66%
Epoch 11/25 - Loss: 0.1247 - Train Acc: 96.68% - Val Acc: 95.74%
Epoch 12/25 - Loss: 0.1154 - Train Acc: 97.00% - Val Acc: 96.00%
Epoch 13/25 - Loss: 0.1065 - Train Acc: 97.34% - Val Acc: 96.28%
Epoch 14/25 - Loss: 0.0983 - Train Acc: 97.55% - Val Acc: 96.39%
Epoch 15/25 - Loss: 0.0910 - Train Acc: 97.76% - Val Acc: 96.55%
E